# Segmented Visual Proprioception Flow (White Background)

Compares **segmented visual proprioception** (robot segmentation filter + white background fill applied to all training, validation, and held-out frames) against **original (unsegmented) visual proprioception** across matched encoder families (`Conv-VAE-Neo`, `ProprioTuned ViT`, and optionally `VGG19` / `ResNet50`) in a single BerryPicker flow.

- **Stage 0:** Imports the demopack (`random-both-cameras-video`) with `group_chooser_sp_vp_standard` and creates a sibling white-background demonstration run (`<demopack>_seg_white`) using a frozen `SmallSegUNet (SAM 3.1)` robot segmenter.
- **Stage 1:** Trains paired `(orig, seg)` visual encoders with identical splits, seeds, and hyperparameters.
- **Stage 2:** Trains paired `(orig, seg)` 6-DoF visual proprioception regressors.
- **Stage 3:** Runs `Compare_VisualProprioception.ipynb` on both strictly held-out `vp_testing` (`heldout_data`, episodes `8..9`, unseen by the segmenter, encoders, and VP regressors) and `vp_validation`, then prints a paired `orig` vs. `seg` RMSE table.

In [ ]:
import copy
import pathlib
import pprint
import sys
import yaml

_cwd = pathlib.Path.cwd().resolve()
for _candidate in (_cwd / "src", _cwd, _cwd / "..", _cwd / "../.."):
    _cand = _candidate.resolve()
    if (_cand / "exp_run_config.py").is_file():
        SRC_ROOT = _cand
        break
else:
    raise RuntimeError(f"Could not locate BerryPicker/src from {_cwd}")
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import torch
import papermill
from IPython.display import Image, Markdown, display

from flow import setup_flow, flow_entry, run_flow, display_flow_report
from demonstration.demopack import import_demopack, group_chooser_sp_vp_standard
from _experimental._2026_11_robot_segmentation import load_segmenter
from _experimental._2026_11_robot_segmentation.segmented_demo import (
    build_segmented_demopack,
    summarize_comparison_csv,
)

print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "| runtime device:", Config().runtime["device"])

## Flow Parameters

Defaults are tuned for a fast laptop smoke verification (`szenes`, Apple M3 Max MPS). For a full CUDA sweep on `glassy`, override via papermill (`-p flow_name SegmentedVP_full_01 -p epochs_sp_vae 100 -p epochs_sp_vit 40 -p epochs_vp 200 ...`).

In [ ]:
flow_name = "SegmentedVP_smoke_01"
demopack_name = "random-both-cameras-video"
demonstration_cam = "dev2"
seg_model_dir = "/Users/lboloni/Documents/Develop/Data/BerryPicker-Flows/RobotSegmentation_pilot/models/unet_sam3_dev2/best"
seg_device = "auto"
latent_sizes = [128]
do_VAE_NEO = True
do_VIT = True
do_VGG = False
do_RESNET = False
vae_image_size = [128, 128]
vae_frame_stride = 2
epochs_sp_vae = 5
epochs_sp_vit = 3
epochs_sp_cnn = 3
epochs_vp = 20
image_size = [256, 256]
creation_style = "exist-ok"
compare_creation_style = "discard-old"
overwrite_seg_demo = False

## Stage 0: Setup Flow, Import Demopack, and Build White-Background Sibling Run

In [ ]:
expruns_path, results_path, notebooks_path = setup_flow(flow_name, [
    "robot_al5d",
    "demonstration",
    "sensorprocessing_conv_vae_neo",
    "sensorprocessing_propriotuned_cnn",
    "sensorprocessing_propriotuned_Vit",
    "visual_proprioception",
    "visual_proprioception_collections",
])

demopack_path = pathlib.Path(Config()["demopacks_path"], demopack_name).expanduser()
if not demopack_path.exists():
    raise FileNotFoundError(f"Demopack not found: {demopack_path}")

selection = import_demopack(demopack_path, group_chooser_sp_vp_standard)
pprint.pprint(selection)

all_episodes = sorted({
    ep
    for split_key in ("sp_training", "sp_validation", "vp_training", "vp_validation", "vp_testing")
    for ep in selection[split_key]
})
seg_demopack_name = f"{demopack_name}_seg_white"

seg_model_path = pathlib.Path(seg_model_dir).expanduser()
if not seg_model_path.exists():
    raise FileNotFoundError(f"Robot segmenter checkpoint not found: {seg_model_path}")

segmenter = load_segmenter(seg_model_path, device=seg_device)
exp_src_demo = Config().get_experiment("demonstration", demopack_name)
src_yaml_path = pathlib.Path(Config().get_exprun_path(), "demonstration", f"{demopack_name}.yaml")
dst_yaml_path = pathlib.Path(Config().get_exprun_path(), "demonstration", f"{seg_demopack_name}.yaml")
dst_demo_root = pathlib.Path(results_path, "demonstration", seg_demopack_name)
preview_path = dst_demo_root / "seg_white_preview.png"
seg_summary = build_segmented_demopack(
    src_yaml_path=src_yaml_path,
    dst_yaml_path=dst_yaml_path,
    src_root=pathlib.Path(exp_src_demo.data_dir()),
    dst_root=dst_demo_root,
    dst_demo_run=seg_demopack_name,
    episodes=all_episodes,
    camera=demonstration_cam,
    segmenter=segmenter,
    preview_path=preview_path,
    overwrite=overwrite_seg_demo,
)
# Register the generated demonstration run with ExpRunFlow
exp_seg_demo = Config().get_experiment("demonstration", seg_demopack_name, creation_style="exist-ok")
pprint.pprint({k: v for k, v in seg_summary.items() if k != "per_episode"})
if preview_path.is_file():
    display(Image(filename=str(preview_path)))

## Stage 1 & 2: Generate Matched `(orig, seg)` SP, VP, and Comparison Exp/Runs

In [ ]:
def make_split_entries(demo_run, episodes, camera):
    return [[demo_run, ep, camera] for ep in episodes]


def write_exprun_yaml(exp_name, run_name, val):
    path = pathlib.Path(Config().get_exprun_path(), exp_name, f"{run_name}.yaml")
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(val, f)
    return path


def generate_sp_conv_vae_neo(params, exp_name, run_name):
    val = copy.copy(params)
    val["class"] = "ConvVaeNeoSensorProcessing"
    val["input-to-notebook"] = [
        "sensorprocessing/Train_Conv_VAE_Neo.ipynb",
        "sensorprocessing/Verify_Conv_VAE_Neo.ipynb",
    ]
    write_exprun_yaml(exp_name, run_name, val)
    return flow_entry(f"Train_{run_name}", exp_name, run_name, 0, creation_style)


def generate_sp_propriotuned_vit(params, exp_name, run_name):
    val = copy.copy(params)
    val["output_size"] = 6
    val["batch_size"] = 32
    val["vit_model"] = "vit_b_16"
    val["vit_output_dim"] = 768
    val["projection_hidden_dim"] = 512
    val["vit_weights"] = "DEFAULT"
    val["proprio_step_1"] = 64
    val["proprio_step_2"] = 64
    val["input-to-notebook"] = ["sensorprocessing/Train_ProprioTuned_VIT.ipynb"]
    write_exprun_yaml(exp_name, run_name, val)
    return flow_entry(f"Train_{run_name}", exp_name, run_name, 0, creation_style)


def generate_sp_propriotuned_cnn(params, exp_name, run_name):
    val = copy.copy(params)
    val["output_size"] = 6
    val["batch_size"] = 32
    write_exprun_yaml(exp_name, run_name, val)
    return flow_entry(f"Train_{run_name}", exp_name, run_name, 0, creation_style)


def generate_vp_train(params, exp_name, run_name):
    val = copy.copy(params)
    val["input-to-notebook"] = [
        "visual_proprioception/Train_VisualProprioception.ipynb",
        "visual_proprioception/Verify_VisualProprioception.ipynb",
    ]
    write_exprun_yaml(exp_name, run_name, val)
    return flow_entry(f"Train_{run_name}", exp_name, run_name, 0, creation_style)


def generate_vp_compare(params, collection_experiment, run_name):
    val = copy.copy(params)
    val["name"] = run_name
    val["source_experiment"] = "visual_proprioception"
    val["input-to-notebook"] = ["visual_proprioception/Compare_VisualProprioception.ipynb"]
    write_exprun_yaml(collection_experiment, run_name, val)
    return flow_entry(f"Compare_{run_name}", collection_experiment, run_name, 0, compare_creation_style)

In [ ]:
variants = [
    ("orig", demopack_name),
    ("seg", seg_demopack_name),
]

expruns = []
sps = []

for latent_size in latent_sizes:
    if do_VAE_NEO:
        for variant_tag, demo_run in variants:
            sp_train = make_split_entries(demo_run, selection["sp_training"], demonstration_cam)
            sp_val = make_split_entries(demo_run, selection["sp_validation"], demonstration_cam)
            exp_name = "sensorprocessing_conv_vae_neo"
            run_name = f"sp_vae_neo_{latent_size}_{variant_tag}"
            params = {
                "latent_size": latent_size,
                "image_size": list(vae_image_size),
                "frame_stride": int(vae_frame_stride),
                "epochs": int(epochs_sp_vae),
                "early_stopping_patience": max(10, int(epochs_sp_vae)),
                "training_data": sp_train,
                "validation_data": sp_val,
            }
            entry = generate_sp_conv_vae_neo(params, exp_name, run_name)
            entry["latent_size"] = latent_size
            entry["variant_tag"] = variant_tag
            entry["demo_run"] = demo_run
            entry["sp_class"] = "ConvVaeNeoSensorProcessing"
            sps.append(entry)
            expruns.append(entry)

    if do_VIT:
        for variant_tag, demo_run in variants:
            sp_train = make_split_entries(demo_run, selection["sp_training"], demonstration_cam)
            sp_val = make_split_entries(demo_run, selection["sp_validation"], demonstration_cam)
            exp_name = "sensorprocessing_propriotuned_Vit"
            run_name = f"sp_vit_base_{latent_size}_{variant_tag}"
            params = {
                "image_size": [224, 224],
                "latent_size": latent_size,
                "epochs": int(epochs_sp_vit),
                "training_data": sp_train,
                "validation_data": sp_val,
                "class": "Vit",
                "model": "VitProprioTunedRegression",
                "freeze_feature_extractor": False,
                "loss": "MSELoss",
                "learning_rate": 0.0001,
            }
            entry = generate_sp_propriotuned_vit(params, exp_name, run_name)
            entry["latent_size"] = latent_size
            entry["variant_tag"] = variant_tag
            entry["demo_run"] = demo_run
            entry["sp_class"] = "VitSensorProcessing"
            sps.append(entry)
            expruns.append(entry)

    for cnntype, enabled in (("vgg19", do_VGG), ("resnet50", do_RESNET)):
        if not enabled:
            continue
        for variant_tag, demo_run in variants:
            sp_train = make_split_entries(demo_run, selection["sp_training"], demonstration_cam)
            sp_val = make_split_entries(demo_run, selection["sp_validation"], demonstration_cam)
            exp_name = "sensorprocessing_propriotuned_cnn"
            run_name = f"sp_{cnntype}_{latent_size}_{variant_tag}"
            params = {
                "image_size": list(image_size),
                "latent_size": latent_size,
                "epochs": int(epochs_sp_cnn),
                "training_data": sp_train,
                "validation_data": sp_val,
                "loss": "MSELoss",
                "learning_rate": 0.001,
            }
            if cnntype == "vgg19":
                params["class"] = "VGG19ProprioTunedSensorProcessing"
                params["model"] = "VGG19ProprioTunedRegression"
            else:
                params["class"] = "ResNetProprioTunedSensorProcessing"
                params["model"] = "ResNetProprioTunedRegression"
                params["freeze_feature_extractor"] = True
                params["reductor_step_1"] = 512
                params["proprio_step_1"] = 64
                params["proprio_step_2"] = 16
            entry = generate_sp_propriotuned_cnn(params, exp_name, run_name)
            entry["latent_size"] = latent_size
            entry["variant_tag"] = variant_tag
            entry["demo_run"] = demo_run
            entry["sp_class"] = params["class"]
            sps.append(entry)
            expruns.append(entry)

# Pair each encoder with its visual proprioception regressor (with heldout_data = vp_testing)
vpruns = []
for sp_entry in sps:
    spexp = sp_entry["experiment"]
    sprun = sp_entry["run"]
    latent_size = sp_entry["latent_size"]
    demo_run = sp_entry["demo_run"]

    vp_run_name = "vp_" + sprun[3:]
    vpruns.append(vp_run_name)
    params = {
        "name": vp_run_name,
        "output_size": 6,
        "encoding_size": latent_size,
        "training_data": make_split_entries(demo_run, selection["vp_training"], demonstration_cam),
        "validation_data": make_split_entries(demo_run, selection["vp_validation"], demonstration_cam),
        "heldout_data": make_split_entries(demo_run, selection["vp_testing"], demonstration_cam),
        "proprioception_heldout_input_file": "heldout_inputs.pt",
        "proprioception_heldout_target_file": "heldout_targets.pt",
        "regressor_hidden_size_1": 64,
        "regressor_hidden_size_2": 64,
        "loss": "MSE",
        "epochs": int(epochs_vp),
        "batch_size": 64,
        "sensor_processing": sp_entry["sp_class"],
        "sp_experiment": spexp,
        "sp_run": sprun,
    }
    expruns.append(generate_vp_train(params, "visual_proprioception", vp_run_name))

# Comparison collections: strictly held-out vp_testing (primary) and vp_validation (reference)
collection_experiment = "visual_proprioception_collections"
expruns.append(
    generate_vp_compare(
        {"tocompare": vpruns, "comparison_dataset": "heldout_data"},
        collection_experiment,
        "vp_comp_heldout",
    )
)
expruns.append(
    generate_vp_compare(
        {"tocompare": vpruns, "comparison_dataset": "validation_data"},
        collection_experiment,
        "vp_comp_val",
    )
)

print(f"Scheduled {len(expruns)} flow stages ({len(vpruns)} VP models):")
for item in expruns:
    print(f"  - {item['name']}: {item['experiment']}/{item['run']}")

## Stage 3: Run the Flow and Summarize `orig` vs. `seg` RMSE

In [ ]:
def stage_already_trained(entry):
    if entry.get("creation_style") != "exist-ok":
        return False
    stage_dir = pathlib.Path(results_path, entry["experiment"], entry["run"])
    return stage_dir.is_dir() and any(stage_dir.glob("*.pth"))

stages_to_run = [e for e in expruns if not stage_already_trained(e)]
skipped = [e["name"] for e in expruns if stage_already_trained(e)]
if skipped:
    print(f"Skipping {len(skipped)} already-trained stages: {skipped}")

flow_error = None
try:
    run_flow(stages_to_run, expruns_path, results_path, notebooks_path)
except Exception as error:
    flow_error = error

In [ ]:
heldout_dir = pathlib.Path(results_path, "visual_proprioception_collections", "vp_comp_heldout")
val_dir = pathlib.Path(results_path, "visual_proprioception_collections", "vp_comp_val")

preview_files = [
    p for p in (heldout_dir / "comparison23.png", heldout_dir / "msecomparison23.png")
    if p.is_file()
]
report = display_flow_report(
    expruns,
    results_path,
    notebooks_path,
    heldout_dir,
    flow_error=flow_error,
    preview_files=preview_files,
)

for label, comp_dir in (("Held-out (`vp_testing`, unseen by segmenter & VP)", heldout_dir), ("Validation (`vp_validation`)", val_dir)):
    csv_file = comp_dir / "all_msecomparison_values.csv"
    if csv_file.is_file():
        _, md_table = summarize_comparison_csv(csv_file)
        display(Markdown(f"### {label}\n\n{md_table}"))

if flow_error is not None:
    raise flow_error
if not report["all-results-present"]:
    raise Exception("Flow finished without all expected stage results.")